In [7]:
import polars as pl
from pathlib import Path

In [8]:
n = 1_000_000

df = pl.select(
    (pl.datetime(2025, 9, 2, 13, 30, 0) + pl.duration(milliseconds=pl.arange(0, n, eager=True))).alias("timestamp"),
    ((pl.arange(0, n, eager=True) % 1000).cast(pl.Float64) + 100.0).alias("price"),
    ((pl.arange(0, n, eager=True) % 50 + 1).cast(pl.Int64)).alias("size"),
)

df.head(), df.shape

(shape: (5, 3)
 ┌─────────────────────────┬───────┬──────┐
 │ timestamp               ┆ price ┆ size │
 │ ---                     ┆ ---   ┆ ---  │
 │ datetime[μs]            ┆ f64   ┆ i64  │
 ╞═════════════════════════╪═══════╪══════╡
 │ 2025-09-02 13:30:00     ┆ 100.0 ┆ 1    │
 │ 2025-09-02 13:30:00.001 ┆ 101.0 ┆ 2    │
 │ 2025-09-02 13:30:00.002 ┆ 102.0 ┆ 3    │
 │ 2025-09-02 13:30:00.003 ┆ 103.0 ┆ 4    │
 │ 2025-09-02 13:30:00.004 ┆ 104.0 ┆ 5    │
 └─────────────────────────┴───────┴──────┘,
 (1000000, 3))

In [9]:
out = Path("data/parquet/demo.parquet")
out.parent.mkdir(parents=True, exist_ok=True)

df.write_parquet(out, compression="zstd")
out.stat().st_size / 1e6

1.873375

In [10]:
result = (
    pl.scan_parquet("data/parquet/demo.parquet")
    .filter(pl.col("price") > 0)
    .group_by_dynamic("timestamp", every="1s")
    .agg(pl.col("size").sum().alias("size_sum"))
    .collect()
)

result.head(10), result.shape

(shape: (10, 2)
 ┌─────────────────────┬──────────┐
 │ timestamp           ┆ size_sum │
 │ ---                 ┆ ---      │
 │ datetime[μs]        ┆ i64      │
 ╞═════════════════════╪══════════╡
 │ 2025-09-02 13:30:00 ┆ 25500    │
 │ 2025-09-02 13:30:01 ┆ 25500    │
 │ 2025-09-02 13:30:02 ┆ 25500    │
 │ 2025-09-02 13:30:03 ┆ 25500    │
 │ 2025-09-02 13:30:04 ┆ 25500    │
 │ 2025-09-02 13:30:05 ┆ 25500    │
 │ 2025-09-02 13:30:06 ┆ 25500    │
 │ 2025-09-02 13:30:07 ┆ 25500    │
 │ 2025-09-02 13:30:08 ┆ 25500    │
 │ 2025-09-02 13:30:09 ┆ 25500    │
 └─────────────────────┴──────────┘,
 (1000, 2))

In [11]:
import time

t0 = time.perf_counter()
(
    pl.scan_parquet("data/parquet/demo.parquet")
    .filter(pl.col("price") > 0)
    .group_by_dynamic("timestamp", every="1s")
    .agg(pl.col("size").sum())
    .collect()
)
print(f"{time.perf_counter() - t0:.3f}s")

0.050s
